# pyalert-runner Tutorial & Feature Walkthrough

A practical guide to using **`pyalert-runner`** for lightweight execution tracking, milestone updates, and automatic crash alerting via your own Google Apps Script bridge.

---

### Core Features Covered:
1. Environment & Connection Test
2. Manual Milestone Checkpoints (`alert.checkpoint`)
3. Smart Cooldown & Throttled Digest Aggregation
4. Artifact & Plot Attachments
5. Tracking Execution Blocks (`alert.track_block`)
6. Crash Alerting with Full Tracebacks

## 1. Setup & Imports

Before running this notebook, verify you have run the configuration wizard in your terminal:
```bash
pyalert-setup --generate-script
pyalert-setup
```

In [2]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pyalert import PyAlert, load_config

# Verify stored settings
cfg = load_config()
print("Webhook Configured:", bool(cfg.webhook_url))
print("Recipient Email:   ", cfg.recipient_email)
print("Config File Path:  ", "~/.config/pyalert/config.json")

Webhook Configured: True
Recipient Email:    sudiptam.iitkgp.phy@gmail.com
Config File Path:   ~/.config/pyalert/config.json


## 2. Test Connection

Send a quick test notification to confirm the webhook endpoint and credentials.

In [3]:
alert = PyAlert(project_name="Tutorial-Demo", cooldown=60)

# Send a synchronous test alert
status = alert.test_connection()
print("Test notification delivered successfully:", status)

[pyalert] INFO: NVIDIA NVML library not found — GPU metrics will be omitted (this is normal on machines without an NVIDIA GPU/driver).


Test notification delivered successfully: True


## 3. Milestone Checkpoints (`alert.checkpoint`)

Use `alert.checkpoint()` to dispatch structured metrics during a run without spamming. Setting `force=True` ensures essential milestone notifications bypass cooldown windows.

In [ ]:
alert = PyAlert(project_name="Simulation-Grid", cooldown=60)

for epoch in range(1, 101):
    time.sleep(0.01)  # Simulated computation
    
    # Alert strictly at milestone epochs
    if epoch % 50 == 0:
        alert.checkpoint(
            message=f"Completed Epoch {epoch}/100",
            level="INFO",
            extra={
                "Epoch": epoch,
                "Mean Loss": f"{0.5 / epoch:.4f}",
                "Convergence Ratio": f"{0.95 + (epoch * 0.0004):.4f}"
            },
            force=True
        )

print("Milestones completed.")

[pyalert] INFO: NVIDIA NVML library not found — GPU metrics will be omitted (this is normal on machines without an NVIDIA GPU/driver).


Milestones completed.


## 4. Smart Cooldown & Throttled Digest Aggregation

When multiple checkpoints fire within the cooldown window (`cooldown=30`), `pyalert` buffers them client-side. Calling `alert.flush()` coalesces the buffered events into a single digest email.

In [7]:
alert_buffered = PyAlert(project_name="Fast-Loop-Test", cooldown=30)

print("Queueing 3 checkpoints in rapid succession...")
for i in range(1, 4):
    alert_buffered.checkpoint(
        message=f"Batch step {i} finished",
        level="INFO",
        extra={"Step": i, "Score": round(i * 12.4, 2)}
    )
    time.sleep(0.2)

print("Flushing buffer into a single combined digest email...")
alert_buffered.flush(wait=True)
print("Batch digest dispatched!")

[pyalert] INFO: NVIDIA NVML library not found — GPU metrics will be omitted (this is normal on machines without an NVIDIA GPU/driver).


Queueing 3 checkpoints in rapid succession...
Flushing buffer into a single combined digest email...
Batch digest dispatched!


## 5. Artifact & Plot Attachments

Pass file paths to `attachments=[...]` to base64-encode and deliver plots, CSV files, or logs directly to your email.

In [8]:
# Create a sample optimization curve
x = np.linspace(0, 10, 100)
y = np.exp(-0.3 * x) * np.cos(2 * np.pi * 0.5 * x)

plot_filename = "convergence_metric.png"
plt.figure(figsize=(6, 3.5))
plt.plot(x, y, label="Approximation Ratio", color="#2563eb")
plt.title("Optimization Trajectory")
plt.xlabel("Iteration")
plt.ylabel("Cost Value")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig(plot_filename, dpi=150)
plt.close()

# Dispatch alert with the artifact attached
alert.checkpoint(
    message="Benchmark completed with artifact plot",
    level="SUCCESS",
    extra={"Final Value": -14.281, "Partition": "Modularity"},
    attachments=[plot_filename],
    force=True
)
print("Alert with image attachment sent.")

Alert with image attachment sent.


## 6. Context-Managed Code Blocks (`alert.track_block`)

Wrap any critical step inside `with alert.track_block("Name"):` to automatically record runtime and system resource metrics upon exit.

In [ ]:
with alert.track_block("Matrix Inversion & SVD"):
    # Simulated numerical operations
    A = np.random.randn(1500, 1500)
    inv = np.linalg.pinv(A)
    time.sleep(0.4)

print("Tracked block finished. Performance and duration metrics dispatched.")

## 7. Crash Alerting with Full Tracebacks

In `try...except` blocks, `alert.report_exception(exc)` immediately bypasses rate limits and sends the full Python stack trace and current memory snapshot.

In [9]:
try:
    print("Executing risky step...")
    result = 100.0 / 0.0  # Intentional error
except Exception as exc:
    print("Caught exception! Dispatching urgent crash report...")
    alert.report_exception(
        exc=exc,
        context="Matrix Normalization Failed"
    )
    print("Crash report dispatched to your inbox.")

Executing risky step...
Caught exception! Dispatching urgent crash report...
Crash report dispatched to your inbox.
